## TRAINING PHASE

Training phase (we use NTU RGB+D for training our Support Vector Machine model because it has action recognition (walking, eating))

In [ ]:
pip install numpy==1.24.4 pandas scikit-learn opencv-python matplotlib jupyter tqdm mediapipe

### Data loading and processing


In [13]:
import numpy as np
import os

class NTUNpyLoader:
    def __init__(self, dataset_path):
        self.dataset_path = dataset_path
        self.joint_count = 25
        self.person_count = 2
        self.coord_dim = 3
        self.max_frames = 300

    def load_npy_dataset(self):
        """Load the .npy files """
        x_train_path = os.path.join(self.dataset_path, 'x_train.npy')
        y_train_path = os.path.join(self.dataset_path, 'y_train.npy')
        x_test_path = os.path.join(self.dataset_path, 'x_test.npy')
        y_test_path = os.path.join(self.dataset_path, 'y_test.npy')

        # Check if files exist
        for path in [x_train_path, y_train_path, x_test_path, y_test_path]:
            if not os.path.exists(path):
                raise FileNotFoundError(f"File not found: {path}")

        print("Loading .npy files...")

        # SIMPLE DIRECT LOADING 
        x_train = np.load(x_train_path)
        y_train = np.load(y_train_path)
        x_test = np.load(x_test_path)
        y_test = np.load(y_test_path)

        print(f"Training data: {x_train.shape}, labels: {y_train.shape}")
        print(f"Testing data: {x_test.shape}, labels: {y_test.shape}")

        # Check if labels are one-hot and convert if needed
        if len(y_train.shape) > 1 and y_train.shape[1] > 1:
            print("Converting one-hot labels to class indices...")
            y_train = np.argmax(y_train, axis=1)
            y_test = np.argmax(y_test, axis=1)

        return (x_train, y_train), (x_test, y_test)

    def reshape_skeleton_sequence(self, sequence):
        """
        Reshape sequence from (300, 150) to (300, 2, 25, 3)
        150 = 2 persons × 25 joints × 3 coordinates
        """
        reshaped = sequence.reshape(self.max_frames, self.person_count, self.joint_count, self.coord_dim)
        return reshaped

    def get_valid_frames(self, sequence):
        """
        Extract valid (non-zero padded) frames from sequence
        """
        # Find frames where there's actual data (not all zeros)
        frame_has_data = np.any(sequence != 0, axis=1)  # Check along feature dimension
        valid_indices = np.where(frame_has_data)[0]

        if len(valid_indices) == 0:
            return sequence[:1]  # Return at least one frame

        return sequence[valid_indices]

    def get_action_names(self):
        """NTU RGB+D 60 action classes"""
        action_names = [
            "drink water", "eat meal", "brush teeth", "brush hair", "drop", "pick up",
            "throw", "sitting down", "standing up", "clapping", "reading", "writing",
            "tear up paper", "wear jacket", "take off jacket", "wear a shoe", "take off a shoe",
            "wear on glasses", "take off glasses", "put on a hat", "take off a hat",
            "cheer up", "hand waving", "kicking something", "reach into pocket",
            "hopping", "jump up", "make a phone call", "playing with phone", "type on a keyboard",
            "point to something", "taking a selfie", "check time (from watch)", "rub two hands",
            "nod head", "shake head", "wipe face", "salute", "put palms together",
            "cross hands in front", "sneeze", "staggering", "falling down", "touch head",
            "touch chest", "touch back", "touch neck", "nausea or vomiting", "use a fan",
            "punching", "kicking", "pushing", "pat on back", "point finger", "hugging",
            "giving something", "touch pocket", "shooting", "walking towards", "walking apart"
        ]
        return action_names

### Feature extraction


In [14]:
from scipy.spatial import distance

class NTUProcessedFeatureExtractor:
    def __init__(self):
        # CORRECT NTU RGB+D 25 Joints
        self.joint_names = [
            "pelvis",       # 1(base of spine)
            "middle_of_spine",      # 2 
            "neck", "head", # 3, 4
            "left_shoulder", "left_elbow", "left_wrist",  # 5, 6, 7
            "left_hand", "right_shoulder", # 8, 9 
            "right_elbow", "right_wrist",  # 10, 11
            "right_hand", "left_hip", # 12, 13
             "left_knee", "left_ankle", "left_foot",  # 14, 15, 16
            "right_hip", "right_knee", "right_ankle", "right_foot",  # 17, 18, 19, 20
            "spine", "tip_left_hand" # 21, 22
            "left_thumb", "tip_right_hand" #23, 24
            "right_thumb" #25
        ]
        
        # CORRECT BONE PAIRS for NTU RGB+D
        self.bone_pairs = [
            # Spine chain (4 bones)
            (1, 0),    # pelvis → middle_of_spine
            (20, 1),    # middle_of_spine → spine  
            (2, 20),    # spine → neck
            (3, 2),    # neck → head
            
            # Left arm chain (5 bones)
            (4, 2),    # neck → left_shoulder (connects to neck, not spine_2)
            (5, 4),    # left_shoulder → left_elbow
            (6, 5),    # left_elbow → left_wrist
            (7, 6),   # left_wrist → left_hand
            (21, 6), # left_wrist -> tip_left_hand
            (22, 6),   # left_wrist → left_thumb
            
            # Right arm chain (5 bones)
            (8, 2),    # neck → right_shoulder
            (9, 8),   # right_shoulder → right_elbow
            (10, 9),  # right_elbow → right_wrist
            (11, 10),  # right_wrist → right_hand
            (23, 10), # right_wrist -> tip_right_hand
            (24, 10),  # right_wrist → right_thumb
            
            # Left leg chain (4 bones)
            (12, 0),   # pelvis → left_hip
            (13, 12),  # left_hip → left_knee
            (14, 13),  # left_knee → left_ankle
            (15, 14),  # left_ankle → left_foot
            
            # Right leg chain (4 bones)
            (16, 0),   # pelvis → right_hip  
            (17, 16),  # right_hip → right_knee
            (18, 17),  # right_knee → right_ankle
            (19, 18),  # right_ankle → right_foot
        ]

        '''
            self.bone_pairs = [
            # Spine chain (4 bones)
            (2, 1),    # pelvis → middle_of_spine
            (21, 2),   # middle_of_spine → spine  
            (3, 21),   # spine → neck
            (4, 3),    # neck → head
            
            # Left arm chain (6 bones)
            (5, 3),    # neck → left_shoulder
            (6, 5),    # left_shoulder → left_elbow
            (7, 6),    # left_elbow → left_wrist
            (8, 7),    # left_wrist → left_hand
            (22, 7),   # left_wrist → tip_left_hand
            (23, 7),   # left_wrist → left_thumb
            
            # Right arm chain (6 bones)
            (9, 3),    # neck → right_shoulder
            (10, 9),   # right_shoulder → right_elbow
            (11, 10),  # right_elbow → right_wrist
            (12, 11),  # right_wrist → right_hand
            (24, 11),  # right_wrist → tip_right_hand
            (25, 11),  # right_wrist → right_thumb
            
            # Left leg chain (4 bones)
            (13, 1),   # pelvis → left_hip
            (14, 13),  # left_hip → left_knee
            (15, 14),  # left_knee → left_ankle
            (16, 15),  # left_ankle → left_foot
            
            # Right leg chain (4 bones)
            (17, 1),   # pelvis → right_hip  
            (18, 17),  # right_hip → right_knee
            (19, 18),  # right_knee → right_ankle
            (20, 19),  # right_ankle → right_foot
        ]
        
        '''
    def extract_features_from_sequence(self, sequence):
        """
        Extract features from preprocessed NTU sequence
        sequence shape: (300, 150) -> will be reshaped to (300, 2, 25, 3)
        """
        # Reshape sequence
        reshaped_sequence = sequence.reshape(300, 2, 25, 3)

        # Use only the first person (main subject)
        main_person_sequence = reshaped_sequence[:, 0, :, :]  # (300, 25, 3)

        # Remove zero-padded frames
        valid_frames = self.get_valid_frames(main_person_sequence)

        if len(valid_frames) == 0:
            return np.array([])

        features = []

        # Extract spatial features from each valid frame
        spatial_features = []
        for frame in valid_frames:
            frame_feat = self.extract_spatial_features(frame)
            spatial_features.append(frame_feat)

        spatial_features = np.array(spatial_features)

        # Extract temporal features
        temporal_features = self.extract_temporal_features(spatial_features)
        features.extend(temporal_features)

        # Statistical features across sequence
        statistical_features = self.extract_statistical_features(spatial_features)
        features.extend(statistical_features)

        return np.array(features)

    def get_valid_frames(self, sequence):
        """Extract non-zero padded frames"""
        valid_frames = []
        for frame in sequence:
            # Check if frame has meaningful data (not all zeros)
            if np.any(frame != 0):
                valid_frames.append(frame)
        return np.array(valid_frames)

    def extract_spatial_features(self, skeleton_frame):
        """Extract spatial features from a single frame"""
        features = []

        # 1. Flatten joint coordinates
        features.extend(skeleton_frame.flatten())

        # 2. Bone vectors between connected joints - WITH SAFETY CHECKS
        for j1, j2 in self.bone_pairs:
            if (j1 < len(skeleton_frame) and j2 < len(skeleton_frame) and
                np.any(skeleton_frame[j1] != 0) and np.any(skeleton_frame[j2] != 0)):
                
                bone_vector = skeleton_frame[j2] - skeleton_frame[j1]
                features.extend(bone_vector)
                # Bone length
                bone_length = np.linalg.norm(bone_vector)
                features.append(bone_length)
            else:
                # Pad with zeros for missing bones
                features.extend([0, 0, 0, 0])

        # 3. Key joint distances from spine base (joint 0) 
        key_joints = [3, 7, 11, 15, 19]  # Head, left_hand, right_hand, left_foot, right_foot
        spine_base = skeleton_frame[0]
        for joint_idx in key_joints:
            if (joint_idx < len(skeleton_frame) and 
                np.any(skeleton_frame[joint_idx] != 0) and 
                np.any(spine_base != 0)):
                
                distance_vec = skeleton_frame[joint_idx] - spine_base
                features.extend(distance_vec)
                features.append(np.linalg.norm(distance_vec))
            else:
                features.extend([0, 0, 0, 0])

        # 4. Joint angles for major limbs 
        angles = self.calculate_joint_angles(skeleton_frame)
        features.extend(angles)

        return np.array(features)

    def calculate_joint_angles(self, skeleton):
        """Calculate joint angles for major limbs """
        angles = []

        # Left elbow angle (shoulder-elbow-wrist)
        # Joints: left_shoulder(4), left_elbow(5), left_wrist(6)
        if (len(skeleton) > 6 and 
            np.any(skeleton[4] != 0) and np.any(skeleton[5] != 0) and np.any(skeleton[6] != 0)):
            left_shoulder = skeleton[4]    
            left_elbow = skeleton[5]        
            left_wrist = skeleton[6]       
            angle = self.calculate_angle(left_shoulder, left_elbow, left_wrist)
            angles.append(angle)
        else:
            angles.append(0.0)

        # Right elbow angle  
        # Joints: right_shoulder(8), right_elbow(9), right_wrist(10)
        if (len(skeleton) > 10 and 
            np.any(skeleton[8] != 0) and np.any(skeleton[9] != 0) and np.any(skeleton[10] != 0)):
            right_shoulder = skeleton[8]  
            right_elbow = skeleton[9]      
            right_wrist = skeleton[10]     
            angle = self.calculate_angle(right_shoulder, right_elbow, right_wrist)
            angles.append(angle)
        else:
            angles.append(0.0)

        # Left knee angle
        # Joints: left_hip(12), left_knee(13), left_ankle(14)
        if (len(skeleton) > 14 and 
            np.any(skeleton[12] != 0) and np.any(skeleton[13] != 0) and np.any(skeleton[14] != 0)):
            left_hip = skeleton[12]        
            left_knee = skeleton[13]       
            left_ankle = skeleton[14]     
            angle = self.calculate_angle(left_hip, left_knee, left_ankle)
            angles.append(angle)
        else:
            angles.append(0.0)

        # Right knee angle
        # Joints: right_hip(16), right_knee(17), right_ankle(18)
        if (len(skeleton) > 18 and 
            np.any(skeleton[16] != 0) and np.any(skeleton[17] != 0) and np.any(skeleton[18] != 0)):
            right_hip = skeleton[16]      
            right_knee = skeleton[17]      
            right_ankle = skeleton[18]     
            angle = self.calculate_angle(right_hip, right_knee, right_ankle)
            angles.append(angle)
        else:
            angles.append(0.0)

        return angles

    def calculate_angle(self, point1, point2, point3):
        """Calculate angle between three points"""
        vec1 = point1 - point2
        vec2 = point3 - point2

        if (np.linalg.norm(vec1) > 0.001 and np.linalg.norm(vec2) > 0.001 and
            np.any(vec1 != 0) and np.any(vec2 != 0)):
            
            cosine_angle = np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
            cosine_angle = np.clip(cosine_angle, -1, 1)
            return np.arccos(cosine_angle)
        return 0.0

    def extract_temporal_features(self, frame_features):
        """Extract temporal features across frames"""
        features = []

        if len(frame_features) > 1:
            # Velocities (frame-to-frame differences)
            velocities = np.diff(frame_features, axis=0)

            # Mean velocity
            if len(velocities) > 0:
                mean_velocity = np.mean(velocities, axis=0)
                features.extend(mean_velocity)

                # Velocity statistics
                velocity_magnitudes = np.linalg.norm(velocities, axis=1)
                features.append(np.mean(velocity_magnitudes))
                features.append(np.std(velocity_magnitudes))
                features.append(np.max(velocity_magnitudes))

        return features

    def extract_statistical_features(self, frame_features):
        """Extract statistical features across sequence"""
        features = []

        if len(frame_features) > 0:
            # Mean positions
            mean_positions = np.mean(frame_features, axis=0)
            features.extend(mean_positions)

            # Standard deviation
            std_positions = np.std(frame_features, axis=0)
            features.extend(std_positions)

            # Range (max-min)
            range_positions = np.ptp(frame_features, axis=0)
            features.extend(range_positions)

            # Motion intensity (overall variance)
            overall_variance = np.mean(std_positions)
            features.append(overall_variance)

        return features

### Main function for training

In [15]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
import pickle
from tqdm import tqdm

class NTUNpyTrainer:
    def __init__(self, dataset_path):
        self.dataset_path = dataset_path
        self.data_loader = NTUNpyLoader(dataset_path)  # Use NTUNpyLoader for .npy files
        self.feature_extractor = NTUProcessedFeatureExtractor()
        self.scaler = StandardScaler()
        
    def prepare_features(self, x_data, y_data, max_samples=None):
        """Extract features from raw sequences"""
        if max_samples and max_samples < len(x_data):
            x_data = x_data[:max_samples]
            y_data = y_data[:max_samples]
        
        print("Extracting features...")
        X_features = []
        y_filtered = []
        
        for i in tqdm(range(len(x_data))):
            try:
                features = self.feature_extractor.extract_features_from_sequence(x_data[i])
                if len(features) > 50:  # Ensure we have meaningful features
                    X_features.append(features)
                    y_filtered.append(y_data[i])
            except Exception as e:
                if i % 500 == 0:  # Print occasional errors
                    print(f"Error processing sample {i}: {e}")
                continue
        
        print(f"Successfully processed {len(X_features)}/{len(x_data)} samples")
        
        # Handle variable length features by padding
        X_padded = self.pad_features(X_features)
        
        return X_padded, np.array(y_filtered)
    
    def pad_features(self, features_list):
        """Pad feature vectors to same length"""
        if len(features_list) == 0:
            return np.array([])
            
        # Find maximum length
        lengths = [len(x) for x in features_list]
        max_len = max(lengths)
        min_len = min(lengths)
        
        print(f"Feature lengths - Min: {min_len}, Max: {max_len}")
        
        # Pad sequences
        X_padded = np.zeros((len(features_list), max_len))
        for i, seq in enumerate(features_list):
            X_padded[i, :len(seq)] = seq
        
        return X_padded
    
    def train_model(self, X_train, y_train, X_test, y_test):
        """Train SVM model"""
        print("Scaling features...")
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        print("Training SVM...")
        # Use simpler parameters for initial training
        svm = SVC(
            kernel='rbf',
            C=1.0,
            gamma='scale',
            probability=True,
            random_state=42,
            verbose=True
        )
        
        svm.fit(X_train_scaled, y_train)
        
        # Evaluate
        train_score = svm.score(X_train_scaled, y_train)
        test_score = svm.score(X_test_scaled, y_test)
        
        print(f" Train Accuracy: {train_score:.3f}")
        print(f" Test Accuracy: {test_score:.3f}")
        
        return svm
    
    def evaluate_model(self, model, X_test, y_test, action_names):
        """Comprehensive evaluation"""
        X_test_scaled = self.scaler.transform(X_test)
        
        y_pred = model.predict(X_test_scaled)
        accuracy = accuracy_score(y_test, y_pred)
        
        print(f"Final Test Accuracy: {accuracy:.3f}")
        print(f"Test Samples: {len(y_test)}")
        print(f"Number of classes: {len(np.unique(y_test))}")
        
        # Show performance for a few sample classes
        unique_classes = np.unique(y_test)
        if len(unique_classes) <= 10:
            sample_classes = unique_classes
        else:
            sample_classes = np.random.choice(unique_classes, 10, replace=False)
        
        print(f"\n Sample classes performance:")
        for class_id in sample_classes:
            class_mask = y_test == class_id
            if np.sum(class_mask) > 0:
                class_accuracy = accuracy_score(y_test[class_mask], y_pred[class_mask])
                class_name = action_names[class_id] if class_id < len(action_names) else f"Class {class_id}"
                print(f"  {class_name}: {class_accuracy:.3f} ({np.sum(class_mask)} samples)")
        
        return accuracy
    
    def save_model(self, model, model_path='models/ntu_npy'):
        """Save trained model"""
        os.makedirs(model_path, exist_ok=True)
        
        with open(os.path.join(model_path, 'svm_model.pkl'), 'wb') as f:
            pickle.dump(model, f)
        
        with open(os.path.join(model_path, 'feature_scaler.pkl'), 'wb') as f:
            pickle.dump(self.scaler, f)
        
        # Save action names
        action_names = self.data_loader.get_action_names()
        with open(os.path.join(model_path, 'action_names.pkl'), 'wb') as f:
            pickle.dump(action_names, f)
        
        print(f" Model saved to {model_path}")

### Training function

In [16]:
def train():
    # Update this path to your downloaded dataset
    dataset_path = "data"

    # Initialize trainer with NTUNpyTrainer for .npy files
    trainer = NTUNpyTrainer(dataset_path)  

    # Load the dataset
    (x_train, y_train), (x_test, y_test) = trainer.data_loader.load_npy_dataset()

    # Get action names
    action_names = trainer.data_loader.get_action_names()
    print(f"Action classes: {len(action_names)}")
    print(f"Full dataset - Training: {x_train.shape}, Test: {x_test.shape}")

    print("\n Preparing training features...")
    
    # Use at least 10,000 training samples (25% of your data)
    X_train_feat, y_train_feat = trainer.prepare_features(x_train, y_train, max_samples=10000)
    
    # Use at least 2,000 test samples (12% of your data)  
    X_test_feat, y_test_feat = trainer.prepare_features(x_test, y_test, max_samples=2000)

    print(f"Final training set: {X_train_feat.shape}")
    print(f"Final test set: {X_test_feat.shape}")
    print(f"Classes in training: {len(np.unique(y_train_feat))}")

    # Check class distribution
    unique, counts = np.unique(y_train_feat, return_counts=True)
    print(f"Samples per class - Min: {np.min(counts)}, Max: {np.max(counts)}, Avg: {np.mean(counts):.1f}")

    # Train model
    model = trainer.train_model(X_train_feat, y_train_feat, X_test_feat, y_test_feat)

    # Evaluate
    trainer.evaluate_model(model, X_test_feat, y_test_feat, action_names)

    # Save model
    trainer.save_model(model)

    print("Training completed successfully!")

In [ ]:
train()

# INFERENCE PHASE

### MediaPipe skeleton to NTU Format converter

In [18]:
import mediapipe as mp
import cv2

class MediaPipeToNTUConverter:
    def __init__(self):
        self.mp_pose = mp.solutions.pose
        self.pose = self.mp_pose.Pose(
            static_image_mode=False,
            model_complexity=1,
            smooth_landmarks=True,
            enable_segmentation=False,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )

        # MediaPipe to NTU joint mapping
        self.joint_mapping = self.create_joint_mapping()

    def create_joint_mapping(self):
        """
        CORRECT mapping from MediaPipe's 33 landmarks to NTU's 25 joints
        """
        return {
            # SPINE CHAIN (5 joints)
            0: 23,   # pelvis (0) -> LEFT_HIP (23) - using as base
            1: 24,   # middle_of_spine (1) -> RIGHT_HIP (24) - approximation
            20: 23,  # spine (20) -> LEFT_HIP (23) - approximation from hip
            2: 11,   # neck (2) -> midpoint between shoulders (calculated)
            3: 0,    # head (3) -> NOSE (0)
            
            # LEFT ARM (6 joints)
            4: 11,   # left_shoulder (4) -> LEFT_SHOULDER (11)
            5: 13,   # left_elbow (5) -> LEFT_ELBOW (13)
            6: 15,   # left_wrist (6) -> LEFT_WRIST (15)
            7: 19,   # left_hand (7) -> LEFT_PINKY (19) - approximation
            21: 17,  # tip_left_hand (21) -> LEFT_INDEX (17)
            22: 21,  # left_thumb (22) -> LEFT_THUMB (21)
            
            # RIGHT ARM (6 joints)
            8: 12,   # right_shoulder (8) -> RIGHT_SHOULDER (12)
            9: 14,   # right_elbow (9) -> RIGHT_ELBOW (14)
            10: 16,  # right_wrist (10) -> RIGHT_WRIST (16)
            11: 20,  # right_hand (11) -> RIGHT_PINKY (20) - approximation
            23: 18,  # tip_right_hand (23) -> RIGHT_INDEX (18)
            24: 22,  # right_thumb (24) -> RIGHT_THUMB (22)
            
            # LEFT LEG (4 joints)
            12: 23,  # left_hip (12) -> LEFT_HIP (23)
            13: 25,  # left_knee (13) -> LEFT_KNEE (25)
            14: 27,  # left_ankle (14) -> LEFT_ANKLE (27)
            15: 31,  # left_foot (15) -> LEFT_FOOT_INDEX (31)
            
            # RIGHT LEG (4 joints)
            16: 24,  # right_hip (16) -> RIGHT_HIP (24)
            17: 26,  # right_knee (17) -> RIGHT_KNEE (26)
            18: 28,  # right_ankle (18) -> RIGHT_ANKLE (28)
            19: 32,  # right_foot (19) -> RIGHT_FOOT_INDEX (32)
        }

    def mediapipe_to_ntu_skeleton(self, landmarks):
        """
        Convert MediaPipe landmarks to NTU-style skeleton (25 joints, 3 coordinates)
        """
        ntu_skeleton = np.zeros((25, 3))

        for ntu_joint, mp_landmark_idx in self.joint_mapping.items():
            if mp_landmark_idx < len(landmarks.landmark):
                landmark = landmarks.landmark[mp_landmark_idx]
                ntu_skeleton[ntu_joint] = [landmark.x, landmark.y, landmark.z]

        return ntu_skeleton

    def process_frame(self, frame):
        """Process a frame and return MediaPipe results"""
        # Convert BGR to RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = self.pose.process(rgb_frame)
        return results

### RealTime Inference

In [19]:
import time
from collections import deque

class RealTimePoseClassifier:
    def __init__(self, model_path='models/ntu_npy'):
        # Load trained model and artifacts with proper error handling
        print("Loading trained model...")
        
        try:
            # First, check if the model path exists
            if not os.path.exists(model_path):
                print(f"Model path '{model_path}' does not exist!")
                print("Available directories:")
                for item in os.listdir('.'):
                    if os.path.isdir(item):
                        print(f"  - {item}")
                raise FileNotFoundError(f"Model path '{model_path}' not found")
            
            print(f"Model path exists: {model_path}")
            print(f"Files in model directory: {os.listdir(model_path)}")
            
            # Load model files one by one with error handling
            model_file = os.path.join(model_path, 'svm_model.pkl')
            scaler_file = os.path.join(model_path, 'feature_scaler.pkl')
            names_file = os.path.join(model_path, 'action_names.pkl')
            
            print("Loading SVM model...")
            with open(model_file, 'rb') as f:
                self.svm_model = pickle.load(f)
            print("SVM model loaded")
            
            print("Loading feature scaler...")
            with open(scaler_file, 'rb') as f:
                self.scaler = pickle.load(f)
            print("Feature scaler loaded")
            
            print("Loading action names...")
            with open(names_file, 'rb') as f:
                self.action_names = pickle.load(f)
            print("Action names loaded")
            
        except Exception as e:
            print(f"Error loading model: {e}")
            raise

        # Initialize converters
        print("Initializing MediaPipe converter...")
        self.converter = MediaPipeToNTUConverter()
        
        print("Initializing feature extractor...")
        self.feature_extractor = NTUProcessedFeatureExtractor()

        # Buffer for temporal features
        self.pose_buffer = deque(maxlen=30)
        self.prediction_buffer = deque(maxlen=10)
        self.fps_history = deque(maxlen=30)

        print(f"Model loaded successfully!")
        print(f"Action classes: {len(self.action_names)}")
        print(f"Sample actions: {self.action_names[:5]}...") 

    def extract_realtime_features(self, ntu_skeleton):
        """
        Extract features compatible with our trained model for real-time inference
        """
        # Add to pose buffer
        self.pose_buffer.append(ntu_skeleton)

        if len(self.pose_buffer) < 5:  # Wait for enough frames
            return None

        # Convert buffer to sequence format
        sequence = np.array(self.pose_buffer)  # (frames, 25, 3)

        # Since we need (300, 150) format for our feature extractor,
        # we'll create a mock sequence that matches the expected format
        mock_sequence = self.create_mock_sequence(sequence)

        # Extract features using our trained feature extractor
        features = self.feature_extractor.extract_features_from_sequence(mock_sequence)

        return features

    def create_mock_sequence(self, real_sequence):
        """
        Create a mock sequence in the format expected by our feature extractor
        (300, 150) where 150 = 2 persons × 25 joints × 3 coordinates
        """
        # Create a 300-frame sequence with our real data
        mock_sequence = np.zeros((300, 150))

        # Fill the beginning with our real data
        num_real_frames = len(real_sequence)
        if num_real_frames > 0:
            # Reshape real sequence to match expected format
            # We only have one person, so we'll put it in the first person slot
            for i in range(min(num_real_frames, 300)):
                # First person data (75 features: 25 joints × 3 coordinates)
                person1_data = real_sequence[i].flatten()
                mock_sequence[i, :75] = person1_data[:75]  # Ensure correct length

                # Second person remains zeros (we only track one person)
                # mock_sequence[i, 75:] = 0  # Already zeros

        return mock_sequence

    def predict_action(self, features):
        """Make prediction using SVM model"""
        if features is None or len(features) == 0:
            return None, 0.0

        # Scale features
        features_scaled = self.scaler.transform([features])

        # Predict
        prediction = self.svm_model.predict(features_scaled)[0]
        probabilities = self.svm_model.predict_proba(features_scaled)[0]
        confidence = np.max(probabilities)

        return prediction, confidence

    def smooth_prediction(self, current_pred, current_confidence):
        """Apply temporal smoothing to predictions"""
        if current_pred is None:
            if len(self.prediction_buffer) > 0:
                # Return most common prediction from buffer
                predictions = [p[0] for p in self.prediction_buffer]
                most_common = max(set(predictions), key=predictions.count)
                avg_confidence = np.mean([p[1] for p in self.prediction_buffer if p[0] == most_common])
                return most_common, avg_confidence
            else:
                return None, 0.0

        # Add current prediction to buffer
        self.prediction_buffer.append((current_pred, current_confidence))

        if len(self.prediction_buffer) < 3:  # Need minimum buffer size
            return current_pred, current_confidence

        # Get most common prediction from buffer
        predictions = [p[0] for p in self.prediction_buffer]
        most_common = max(set(predictions), key=predictions.count)

        # Calculate average confidence for the most common prediction
        common_confidences = [p[1] for p in self.prediction_buffer if p[0] == most_common]
        avg_confidence = np.mean(common_confidences)

        return most_common, avg_confidence

    def draw_skeleton(self, image, landmarks):
        """Draw MediaPipe skeleton on image"""
        mp.solutions.drawing_utils.draw_landmarks(
            image,
            landmarks,
            mp.solutions.pose.POSE_CONNECTIONS,
            mp.solutions.drawing_styles.get_default_pose_landmarks_style()
        )

    def draw_prediction(self, image, prediction, confidence, fps):
        """Draw prediction and info on image"""
        if prediction is not None and confidence > 0.3:  # Confidence threshold
            action_name = self.action_names[prediction]

            # Draw prediction box
            cv2.rectangle(image, (10, 10), (400, 100), (0, 0, 0), -1)
            cv2.rectangle(image, (10, 10), (400, 100), (255, 255, 255), 2)

            # Draw prediction text
            cv2.putText(image, f"Action: {action_name}", (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            cv2.putText(image, f"Confidence: {confidence:.2f}", (20, 70),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            cv2.putText(image, f"FPS: {fps:.1f}", (20, 95),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        else:
            cv2.putText(image, "No pose detected", (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    def run_realtime_demo(self):
        """Main real-time inference loop with better error handling"""
        print("🎥 Initializing webcam...")
        
        # Test webcam access
        cap = cv2.VideoCapture(0)
        if not cap.isOpened():
            print("Cannot access webcam. Trying alternative camera indices...")
            
            # Try different camera indices
            for i in range(1, 5):
                cap = cv2.VideoCapture(i)
                if cap.isOpened():
                    print(f"✅ Found webcam at index {i}")
                    break
                cap.release()
            else:
                print("No webcam found. Please check your camera connection.")
                return
        
        # Set camera properties
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        
        # Test camera read
        ret, test_frame = cap.read()
        if not ret:
            print(" Cannot read from webcam")
            cap.release()
            return
        else:
            print("Webcam is working")
        
        print("Starting real-time pose classification...")
        print("Press 'q' to quit, 'r' to reset buffer")

        last_time = time.time()
        frame_count = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                print("Failed to capture frame")
                break

            # Calculate FPS
            current_time = time.time()
            frame_count += 1
            if current_time - last_time >= 1.0:
                fps = frame_count / (current_time - last_time)
                self.fps_history.append(fps)
                frame_count = 0
                last_time = current_time
                avg_fps = np.mean(self.fps_history) if self.fps_history else 0
            else:
                avg_fps = np.mean(self.fps_history) if self.fps_history else 0

            try:
                # Process frame with MediaPipe
                results = self.converter.process_frame(frame)

                if results.pose_landmarks:
                    # Draw skeleton
                    self.draw_skeleton(frame, results.pose_landmarks)

                    # Convert to NTU format
                    ntu_skeleton = self.converter.mediapipe_to_ntu_skeleton(results.pose_landmarks)

                    # Extract features
                    features = self.extract_realtime_features(ntu_skeleton)

                    # Make prediction
                    if features is not None:
                        current_pred, current_conf = self.predict_action(features)
                        final_pred, final_conf = self.smooth_prediction(current_pred, current_conf)
                        
                        if final_pred is not None:
                            action_name = self.action_names[final_pred]
                            print(f"Prediction: {action_name} (confidence: {final_conf:.2f}, FPS: {avg_fps:.1f})")
                    else:
                        final_pred, final_conf = None, 0.0
                else:
                    final_pred, final_conf = None, 0.0
                    # Clear buffer when no pose is detected
                    self.pose_buffer.clear()

                # Draw prediction
                self.draw_prediction(frame, final_pred, final_conf, avg_fps)

            except Exception as e:
                print(f"Error in processing frame: {e}")
                final_pred, final_conf = None, 0.0

            # Display frame
            cv2.imshow('Real-time Pose Classification', frame)

            # Handle key presses
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                break
            elif key == ord('r'):
                # Reset buffers
                self.pose_buffer.clear()
                self.prediction_buffer.clear()
                print("Buffers reset")

        cap.release()
        cv2.destroyAllWindows()
        print("Real-time demo ended")

# Simple test function
def test_inference():
    """Test the inference pipeline without webcam"""
    classifier = RealTimePoseClassifier()
    print("Inference pipeline ready!")
    return classifier


In [ ]:

# Test the inference pipeline
classifier = test_inference()

In [ ]:
# Start real-time demo
classifier.run_realtime_demo()